# Video2Robot - Google Colab

This notebook runs the video-to-robot motion pipeline on Google Colab Pro GPU.

**Pipeline:** Video → PromptHMR (pose extraction) → SMPL-X → GMR → Robot Motion CSV

## Requirements
- Google Colab Pro (GPU runtime)
- Google API Key (for Veo video generation)
- SMPL-X/SMPL registration (for body models)

## 1. Check GPU & Setup Runtime

In [ ]:
# Verify GPU is available
!nvidia-smi

import torch
print(f"\nPyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"CUDA version: {torch.version.cuda}")

## 2. Clone Repository & Install Dependencies

In [ ]:
import os

# Clone the repo (or upload your own)
REPO_URL = "https://github.com/YOUR_USERNAME/video2robot-hackathon.git"  # <-- UPDATE THIS
REPO_DIR = "/content/video2robot-hackathon"

if not os.path.exists(REPO_DIR):
    # Option 1: Clone from GitHub
    # !git clone --recursive {REPO_URL} {REPO_DIR}
    
    # Option 2: Upload from Google Drive
    from google.colab import drive
    drive.mount('/content/drive')
    !cp -r "/content/drive/MyDrive/video2robot-hackathon" {REPO_DIR}
    
os.chdir(REPO_DIR)
print(f"Working directory: {os.getcwd()}")

In [ ]:
# Initialize submodules
!git submodule update --init --recursive

In [ ]:
# Install system dependencies
!apt-get update -qq
!apt-get install -y -qq libsuitesparse-dev ninja-build libgl1 ffmpeg

In [ ]:
# Install PyTorch (Colab already has it, but ensure CUDA version matches)
!pip install torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu121 -q

In [ ]:
# Install main requirements
!pip install -r requirements.txt -q
!pip install -e . -q

In [ ]:
# Install PromptHMR dependencies
!pip install -r third_party/PromptHMR/requirements.txt -q

In [ ]:
# Install PyTorch3D (required for PromptHMR)
import sys
import torch

pyt_version_str = torch.__version__.split("+")[0].replace(".", "")
cuda_version_str = torch.version.cuda.replace(".", "")

!pip install --no-index --no-cache-dir pytorch3d -f https://dl.fbaipublicfiles.com/pytorch3d/packaging/wheels/py310_cu121_pyt240/download.html -q

# If above fails, build from source:
# !pip install "git+https://github.com/facebookresearch/pytorch3d.git@stable"

In [ ]:
# Install chumpy (needed for SMPL)
!pip install git+https://github.com/mattloper/chumpy@9b045ff5d6588a24a0bab52c83f032e2ba433e17 -q

## 3. Download Checkpoints & Data

In [ ]:
# Download pretrained models from HuggingFace
import os

DATA_ZIP_URL = "https://huggingface.co/daviddobas/video2robot-hackathon-data/resolve/main/video2robot_data.zip"
PHMR_DATA_DIR = "third_party/PromptHMR/data"

if not os.path.exists(f"{PHMR_DATA_DIR}/pretrain"):
    print("Downloading pretrained models...")
    !wget -q {DATA_ZIP_URL} -O /tmp/video2robot_data.zip
    !unzip -q /tmp/video2robot_data.zip -d {PHMR_DATA_DIR}/
    !rm /tmp/video2robot_data.zip
    print("Done!")
else:
    print("Pretrained models already exist.")

In [ ]:
# Create /code/data symlink (required by PromptHMR)
!sudo mkdir -p /code
!sudo ln -sf $(pwd)/third_party/PromptHMR/data /code/data
!ls -la /code/

## 4. Download SMPL Body Models (Required)

You need to register at:
- **SMPL-X**: https://smpl-x.is.tue.mpg.de/register.php
- **SMPL**: https://smpl.is.tue.mpg.de/register.php

Then download and upload the models, or run the fetch script with credentials.

In [ ]:
# Option 1: Upload from Google Drive (if you've already downloaded them)
# Uncomment and adjust path:

# from google.colab import drive
# drive.mount('/content/drive')
# !cp -r "/content/drive/MyDrive/body_models/smplx" third_party/PromptHMR/data/body_models/
# !cp -r "/content/drive/MyDrive/body_models/smpl" third_party/PromptHMR/data/body_models/

In [ ]:
# Option 2: Run fetch script (requires SMPL credentials)
# You'll be prompted for username and password

# !bash scripts/fetch_body_models.sh

In [ ]:
# Verify body models exist
import os

body_models_dir = "third_party/PromptHMR/data/body_models"
required = ["smplx/SMPLX_NEUTRAL.npz", "smpl/SMPL_NEUTRAL.pkl"]

for model in required:
    path = f"{body_models_dir}/{model}"
    if os.path.exists(path):
        print(f"✓ {model}")
    else:
        print(f"✗ {model} - MISSING! Upload body models first.")

## 5. Set API Keys

In [ ]:
import os
from getpass import getpass

# Set Google API Key (for Veo video generation)
# Get your key at: https://aistudio.google.com/apikey
if not os.environ.get("GOOGLE_API_KEY"):
    os.environ["GOOGLE_API_KEY"] = getpass("Enter your Google API Key: ")

print("API key set!")

## 6. Upload Your Video (or Generate with Veo)

In [ ]:
# Option 1: Upload a video file
from google.colab import files
import shutil
import os

PROJECT_NAME = "my_boxing_video"  # Change this!
PROJECT_DIR = f"data/{PROJECT_NAME}"

os.makedirs(PROJECT_DIR, exist_ok=True)

print("Upload your video file:")
uploaded = files.upload()

for filename in uploaded.keys():
    shutil.move(filename, f"{PROJECT_DIR}/original.mp4")
    print(f"Saved to: {PROJECT_DIR}/original.mp4")

In [ ]:
# Option 2: Generate video with Veo (requires Google API key)
# Uncomment to use:

# !python boxing_prompts.py --generate punch_left_jab_face --model veo

## 7. Run the Pipeline

In [ ]:
# Set project path
PROJECT_DIR = "data/punch_left_jab_face"  # <-- Change to your project folder

!ls -la {PROJECT_DIR}/

### Step 1: Extract Pose with PromptHMR

In [ ]:
# Extract human pose from video
!python scripts/extract_pose.py --project {PROJECT_DIR}

### Step 2: Convert to Robot Motion with GMR

In [ ]:
# Convert SMPL-X pose to robot motion
!python scripts/convert_to_robot.py --project {PROJECT_DIR} --robot unitree_g1

### Step 3: Generate LAFAN CSV

In [ ]:
# Convert to CSV format for mjlab
!python scripts/convert_to_lafan_csv.py --project {PROJECT_DIR}

## 8. View Results

In [ ]:
# List output files
import os

print(f"\nProject: {PROJECT_DIR}")
print("="*50)

for f in sorted(os.listdir(PROJECT_DIR)):
    path = os.path.join(PROJECT_DIR, f)
    if os.path.isfile(path):
        size_mb = os.path.getsize(path) / (1024 * 1024)
        print(f"  {f} ({size_mb:.2f} MB)")

In [ ]:
# Preview CSV output
import pandas as pd

csv_path = f"{PROJECT_DIR}/robot_motion.csv"
if os.path.exists(csv_path):
    df = pd.read_csv(csv_path)
    print(f"Shape: {df.shape}")
    print(f"Columns: {list(df.columns[:10])}...")
    display(df.head())
else:
    print(f"CSV not found: {csv_path}")

## 9. Download Results

In [ ]:
# Download robot motion files
from google.colab import files
import os

# Download CSV
csv_path = f"{PROJECT_DIR}/robot_motion.csv"
if os.path.exists(csv_path):
    files.download(csv_path)

# Download pickle
pkl_path = f"{PROJECT_DIR}/robot_motion.pkl"
if os.path.exists(pkl_path):
    files.download(pkl_path)

In [ ]:
# Or save to Google Drive
from google.colab import drive
import shutil

drive.mount('/content/drive')

DRIVE_OUTPUT = "/content/drive/MyDrive/video2robot_outputs"
os.makedirs(DRIVE_OUTPUT, exist_ok=True)

# Copy entire project folder
project_name = os.path.basename(PROJECT_DIR)
shutil.copytree(PROJECT_DIR, f"{DRIVE_OUTPUT}/{project_name}", dirs_exist_ok=True)
print(f"Saved to: {DRIVE_OUTPUT}/{project_name}")

## Batch Processing (All Boxing Actions)

In [ ]:
# Process all boxing actions
ACTIONS = [
    "stance_orthodox",
    "stance_southpaw", 
    "stance_guard",
    "punch_left_jab_face",
    "punch_left_jab_body",
    "punch_right_jab_face",
    "punch_right_jab_body",
    "punch_left_uppercut_face",
    "punch_left_uppercut_body",
    "punch_right_uppercut_face",
    "punch_right_uppercut_body",
    "punch_left_hook_face",
    "punch_left_hook_body",
    "punch_right_hook_face",
    "punch_right_hook_body",
]

import subprocess
import os

for action in ACTIONS:
    project_dir = f"data/{action}"
    video_path = f"{project_dir}/original.mp4"
    
    if not os.path.exists(video_path):
        print(f"\n⏭️  Skipping {action} (no video)")
        continue
    
    print(f"\n{'='*60}")
    print(f"Processing: {action}")
    print(f"{'='*60}")
    
    # Extract pose
    !python scripts/extract_pose.py --project {project_dir}
    
    # Convert to robot
    !python scripts/convert_to_robot.py --project {project_dir} --robot unitree_g1
    
    # Generate CSV
    !python scripts/convert_to_lafan_csv.py --project {project_dir}
    
    print(f"✓ {action} complete!")